# Lyapunov heatmap — trajectories starting on the satellite

Same map as `lyapunov_heatmap.ipynb` (sign of the largest Lyapunov exponent over $\varepsilon_p \times \varepsilon_d/g_2$, same parameters,
`t_max = 30/κ_b`, transient `Ttr = 15/κ_b`). The difference is the initial state: a point of the **satellite** limit cycle, so that a
trajectory that is not chaotic ends on the satellite. Adapted from `analysis/chaos/chiral_cat/comparison_heatmap_satellites.ipynb`.

**Initial state.** The old notebook started from point 75 of a Hill orbit stored for each $\varepsilon_p$ (one $\varepsilon_d/g_2$, older damping,
rotated by hand). Here the satellite is computed for every $(\varepsilon_p, \varepsilon_d/g_2)$ from the analytic harmonic balance
(appendix "Satellites", full system with $u \neq 0$ and the dissipative tilt, as in `limit_cycles.ipynb`), and the trajectory starts at
its $\tau = 0$ point,
$$a(0) = \frac{a_0 + a_1 + a_{-1}}{\varphi_a}, \qquad b(0) = \frac{b_0 + b_{-1} + b_2}{\varphi_b}.$$
The analytic amplitudes are about 10 % below the simulated ones, so the trajectory starts close to the satellite and relaxes onto it if it is stable.
No satellite solution exists (no root with $x^2 > 0$ and $2\kappa_b'|y| \le \varepsilon_d'$) on part of the low-$\varepsilon_p$ region; those points are left blank.

**End state.** Starting near the satellite does not guarantee ending on it, e.g. at $\varepsilon_p = 0.45$, $\varepsilon_d = 10.12\,g_2$ the trajectory
falls onto the cat. Each trajectory is therefore also integrated to $t_\text{max} + T_\text{tr}$ and classified from $\langle|b|\rangle$ over the
last memory period ($\approx 0.1$ on the cat, $\approx 1.6$–$2.1$ on the satellite): satellite, cat, or chaotic ($\lambda > 0$).

In [ ]:
using MPSDynamics, ChaosTools, ProgressBars

In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl")

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 25.338776456203686
ω_b = 2*ω_a


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5


##Drive
ω_d=ω_b
ω_p=2ω_a-ω_d

buffer_phase = 0.0
pump_phase = 0.0;

##Env
κ_b=π/10.4
κ_a = 0

In [ ]:
t_max = 30/κ_b
t_w = 15/κ_b                 # transient discarded by lyapunov (Ttr)
T_a = 2π/ω_a                 # memory period

environment = MeanField(κ_a, κ_b)
system_0 = SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb, 0.0, 0.0, 0.0, 0.0)
environment_prepared = prepare(system_0, environment)
dt = get_dt(system_0, environment)

# Analytic satellite (appendix "Satellites", full system) at (ϵ_p, ϵ_d/g2); member s = ±1 of the pair a → -a.
# Returns the harmonic amplitudes in the paper's units (a_paper = φ_a a, b_paper = φ_b b), or nothing if no root with x² > 0.
function satellite_hb(ϵ_p, ϵ_d_g2_ratio; s = 1, ȳ_grid = range(1e-3, 3.0, length = 3000))
    g2 = besselj(1, ϵ_p)*E_J*φ_a^2*φ_b
    β_a′ = 2E_J*sin(ϵ_p)*φ_a^2/ω_a
    β_b′ = 2E_J*sin(ϵ_p)*φ_b^2/ω_b
    ϵ_d′ = ϵ_d_g2_ratio*g2*φ_b/ω_b
    κ_b′ = κ_b/ω_b
    β_Σ = β_a′ + β_b′
    γ = 1 + β_b′/(β_a′*(1 + κ_b′^2))

    ϵ_eff(y) = sqrt(max(ϵ_d′^2 - 4κ_b′^2*y^2, 0.0))
    function y_of(ȳ)                       # y = ȳ + β_a′ ϵ_eff(y)/(2β_b′), fixed point (κ_b′ ≪ 1)
        y = ȳ + β_a′*ϵ_d′/(2β_b′)
        for _ in 1:30
            y = ȳ + β_a′*ϵ_eff(y)/(2β_b′)
        end
        return y
    end
    u_of(ȳ) = (η = 2β_b′*ȳ; (sqrt((1 + η)^2 + 4η^2) - (1 + η))/(2η))
    x²_of(ȳ) = (u = u_of(ȳ); (2(1 + u)*ȳ^2 - ϵ_eff(y_of(ȳ))/β_b′)/(1 + 2u))
    resid(ȳ) = (u = u_of(ȳ); ȳ*(1 + u) - 2β_Σ*((1 + (1 + u)^2/2)*ȳ^2 + (1 + u^2)*x²_of(ȳ)))

    for k in 1:length(ȳ_grid)-1                # first physical root on the grid, refined by bisection
        lo, hi = ȳ_grid[k], ȳ_grid[k+1]
        resid(lo)*resid(hi) > 0 && continue
        for _ in 1:60
            mid = (lo + hi)/2
            resid(lo)*resid(mid) <= 0 ? (hi = mid) : (lo = mid)
        end
        ȳ = (lo + hi)/2
        x² = x²_of(ȳ)
        x² > 0 || continue
        u, y, x = u_of(ȳ), y_of(ȳ), sqrt(x²)
        2κ_b′*abs(y) <= ϵ_d′ || continue     # phase locking requires 2κ_b′|y| ≤ ϵ_d′
        θ = -asin(2κ_b′*y/ϵ_d′)/2
        a_1 = -(1 + u)*ȳ/(2γ)
        return (a_0 = s*x*cis(θ), a_1 = a_1, a_m1 = -β_a′*ϵ_eff(y)/(2β_b′)*cis(2θ),
                b_0 = y*cis(2θ), b_m1 = s*u*x*cis(3θ), b_2 = β_b′/β_a′*a_1)
    end
    return nothing
end

# laboratory-frame amplitudes at t = 0 (τ = 0), in the simulation's units
satellite_initial_amplitudes(S) = ((S.a_0 + S.a_1 + S.a_m1)/φ_a, (S.b_0 + S.b_m1 + S.b_2)/φ_b)

# SystemATS starting at a(0), b(0)
system_from_amplitudes(a_0, b_0) = SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb,
    sqrt(2*Z_a)*real(a_0), sqrt(2/Z_a)*imag(a_0), sqrt(2*Z_b)*real(b_0), sqrt(2/Z_b)*imag(b_0))

In [ ]:
ϵ_p_list = vcat(0.2:.01:1.1)
ϵ_d_g2_ratio_list = 5:.1:13.5

In [ ]:
using Statistics
Lyap_coef_chaos = fill(NaN, length(ϵ_d_g2_ratio_list), length(ϵ_p_list))   # NaN: no satellite solution
mean_abs_b_end = fill(NaN, length(ϵ_d_g2_ratio_list), length(ϵ_p_list))    # <|b|> over the last memory period
drive = DriveTwoModesBasics(0.0, ω_d, buffer_phase, 0.0, ω_p, pump_phase)
drive_prepared = prepare(drive)
t_end = t_max + t_w
t_save_end = range(t_end - T_a, t_end, length = 60)
for i in ProgressBar(1:length(ϵ_p_list))
    ϵ_p_i = ϵ_p_list[i]
    drive.ϵ_0 = ϵ_p_i
    g2_i = besselj(1,ϵ_p_i)*E_J*φ_a^2*φ_b # (Leon's paper)
    for j in 1:length(ϵ_d_g2_ratio_list)
        S = satellite_hb(ϵ_p_i, ϵ_d_g2_ratio_list[j])
        isnothing(S) && continue
        a_0, b_0 = satellite_initial_amplitudes(S)
        system = system_from_amplitudes(a_0, b_0)
        system_prepared = prepare(system)
        u_dot_0, u_0 = get_init(system_prepared, environment_prepared)

        drive.ϵ_d = ϵ_d_g2_ratio_list[j]*g2_i
        drive_prepared.drive = drive
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
        Lyap_coef_chaos[j,i] = lyapunov(CoupledODEs(equation_1ODE!, vcat(u_0, u_dot_0), p), t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)

        # end state: same trajectory integrated to t_max + Ttr, last memory period only
        ϕ_a, ϕ_b, q_a, q_b, _ = simulator(t_end, system, drive, environment; t_save = t_save_end)
        mean_abs_b_end[j,i] = mean(@. sqrt(ϕ_b^2/(2Z_b) + q_b^2*Z_b/2))
    end
end

In [ ]:
Lyap_coef_chaos_sign = map(x -> isnan(x) ? NaN : (x > 0 ? 1.0 : -1.0), Lyap_coef_chaos)

b_threshold = 0.8   # <|b|> on the last period: ≈ 0.1 on the cat, ≈ 1.6–2.1 on the satellite
# 1 = satellite, 2 = cat, 3 = chaotic, NaN = no satellite solution
end_state = map(Lyap_coef_chaos, mean_abs_b_end) do λ, b
    isnan(λ) ? NaN : (λ > 0 ? 3.0 : (b > b_threshold ? 1.0 : 2.0))
end
n_seed = count(!isnan, end_state)
println("points with a satellite seed: ", n_seed, "/", length(end_state))
for (k, name) in ((1, "satellite"), (2, "cat"), (3, "chaotic"))
    println(rpad(name, 10), round(count(==(k), end_state)/n_seed, digits=3))
end

In [ ]:
using JSON
α_sq = (ϵ_d_g2_ratio_list./2)
g2_list = E_J.*besselj.(1,ϵ_p_list).*φ_a^2 .*φ_b

# instability of the period-2 orbit from Hill_method_doubling_corrected.ipynb (ascending ϵ_d/g2 = 5:13.5, 40 points)
hill_path = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/g2_list_08_10_2026_Hill_unstable.json"
hill_ok = isfile(hill_path)
hill_ok || println("No Hill line: run the 2D sweep and the save cell of Hill_method_doubling_corrected.ipynb to create ", hill_path)

function add_axes!(p)
    hill_ok && plot!(p, Float64.(JSON.parsefile(hill_path)), range(5, 13.5, 40)./2, color=:black, lw=2, label="")
    plot!(p, xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\varepsilon_d/2g_2",
        xlims=(minimum(g2_list/2π*1e3),maximum(g2_list/2π*1e3)), ylims=(minimum(α_sq),maximum(α_sq)))
end

p_lyap = heatmap(g2_list/2π*1e3, α_sq, Lyap_coef_chaos_sign, cmap=:bluesreds, clims=(-1, 1), colorbar=false,
                 title="Sign of Lyapunov exp.")
add_axes!(p_lyap)

end_colors = [RGB(13/256,76/256,116/256), RGB(134/256,187/256,216/256), RGB(185/256,21/256,21/256)]   # satellite, cat, chaotic
p_end = heatmap(g2_list/2π*1e3, α_sq, end_state, color=cgrad(end_colors, 3, categorical=true), clims=(0.5, 3.5),
                colorbar=false, title="End state")
add_axes!(p_end)
for (k, name) in enumerate(("satellite", "cat", "chaotic"))   # legend for the categories
    scatter!(p_end, [NaN], [NaN], color=end_colors[k], markershape=:square, msw=0, label=name)
end
plot!(p_end, legend=:bottomleft)

plot(p_lyap, p_end, layout=(1, 2), size=(2*size_px_car[1], size_px_car[2]), bottom_margin=5mm, left_margin=5mm)
savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/article/heatmap_lyapunov_satellite_start_09_10_2026.pdf")

Same grid and file format as `data_lyapunov_07_04_2026.json` (outer index: ϵ_p). NaN entries are written as `null`.

In [ ]:
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/data_lyapunov_satellite_start_09_10_2026.json", "w") do io
    JSON.print(io, Lyap_coef_chaos_sign)
end
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/end_state_satellite_start_09_10_2026.json", "w") do io
    JSON.print(io, end_state)       # 1 satellite, 2 cat, 3 chaotic, NaN no satellite solution
end
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/g2_list_satellite_start_09_10_2026.json", "w") do io
    JSON.print(io, g2_list)
end
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/alpha_sq_list_satellite_start_09_10_2026.json", "w") do io
    JSON.print(io, α_sq)
end